# NOVA 01b — MOD-01 Obstacle Detection v1.2.0 (COCO baseline)
**No training.** Takes stock COCO-pretrained YOLOv8n (80 real class names), quantizes to INT8 @ 320px, evaluates on COCO128, benchmarks, and publishes to `unixio/nova-obstacle-detection` as v1.2.0.

Rationale: Detectra/VisDrone fine-tunes (v1.0.0/v1.1.0) reached only ~10% mAP50 — kept on the Hub as ablations. The stock model satisfies SRS FR-01-03 ('80-class COCO dataset') with real TTS-speakable names.

**~10 minutes total. No datasets to attach.** GPU + Internet + HF_TOKEN.

In [1]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

Cloning into '/kaggle/working/nova-ml'...


GPU OK: Tesla T4
Bootstrap OK — repo cloned, HF token loaded.


In [2]:
# Pre-install the LiteRT converter stack so Ultralytics' AutoUpdate doesn't
# swap numpy's deps mid-kernel (that corrupts the running Python process).
!pip install -q ultralytics 'litert-torch>=0.9.0' 'ai-edge-litert>=2.1.4' ai-edge-quantizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 79.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 15.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that 

In [3]:
# Evaluate stock YOLOv8n on COCO128 at 320px (deployment resolution).
# COCO128 is small — cite Ultralytics' official full-COCO numbers alongside
# (YOLOv8n: 37.3 mAP50-95 @ 640) in the report.
import json, os
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
metrics = model.val(data='coco128.yaml', imgsz=320)
results = {
    'mAP50_coco128@320': float(metrics.box.map50),
    'mAP50-95_coco128@320': float(metrics.box.map),
    'official_coco_mAP50-95@640': 0.373,
    'note': 'stock COCO-pretrained YOLOv8n, no fine-tune',
}
os.makedirs('/kaggle/working/evaluation', exist_ok=True)
json.dump(results, open('/kaggle/working/evaluation/obstacle_results.json', 'w'), indent=2)
print(json.dumps(results, indent=2))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

WARNING ⚠️ Dataset 'coco128.yaml' images not found, missing path '/kaggle/working/nova-ml/datasets/coco128/images/train2017'
Unzipping /kaggle/working/nova-ml/datasets/coco128.zip to /kaggle/working/nova-ml/datasets/coco128...: 100% ━━━━━━━━━━━━ 263/263 2.8Kfiles/s 0.1s
Dataset download success ✅ (0.3s), saved to /kaggle/working/nova-ml/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1094.8±472.6 MB/s, size: 51.0 KB)
val: Scanning /kaggle/working/nova-ml/datasets/coco128/labels/train2017... 126 images, 2 b

In [4]:
# Write the 80 COCO class names — the app's TTS reads these
os.makedirs('/kaggle/working/labels', exist_ok=True)
names = [model.names[i] for i in range(len(model.names))]
open('/kaggle/working/labels/coco_labels.txt', 'w').write('\n'.join(names) + '\n')
print(len(names), 'classes:', names[:8], '...')

80 classes: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck'] ...


In [5]:
# INT8 quantization at 320 (COCO128 calibration), run in a FRESH subprocess
# via the yolo CLI — immune to any in-kernel package-state issues.
!yolo export model=yolov8n.pt format=litert quantize=int8 imgsz=320 data=coco128.yaml
import glob
candidates = glob.glob('**/yolov8n*int8*.tflite', recursive=True) + \
             glob.glob('**/yolov8n*_int8.tflite', recursive=True)
print('candidates:', candidates)
assert candidates, 'INT8 TFLite not found after export'
exported = candidates[0]
size_mb = os.path.getsize(exported) / 1e6
print(f'{exported}: {size_mb:.2f} MB (budget: <15 MB)')
assert size_mb < 15

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 84, 2100) (6.2 MB)
LiteRT: collecting INT8 calibration images from 'data=coco128.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1175.0±373.0 MB/s, size: 51.3 KB)
val: Scanning /kaggle/working/nova-ml/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 23.3Mit/s 0.0s
WARNING ⚠️ LiteRT: >300 images recommended for INT8 calibration, found 128 images.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to 

In [6]:
# Publish v1.2.0
!python scripts/push_to_huggingface.py --module MOD-01 \
    --tflite {exported} \
    --labels /kaggle/working/labels/coco_labels.txt \
    --eval-json /kaggle/working/evaluation/obstacle_results.json --version 1.2.0

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  yolov8n_int8.tflite         :  97%|█████████████▌| 3.36MB / 3.47MB            

Processing Files (0 / 1)      :  97%|█████████████▌| 3.36MB / 3.47MB, 16.9MB/s  
New Data Upload               :  97%|█████████████▌| 3.36MB / 3.47MB, 16.9MB/s  

Processing Files (1 / 1)      : 100%|██████████████| 3.47MB / 3.47MB, 8.69MB/s  
New Data Upload               : 100%|██████████████| 3.47MB / 3.47MB, 8.69MB/s  

  yolov8n_int8.tflite         : 100%|██████████████| 3.47MB / 3.47MB            

Processing Files (1 / 1)      : 100%|██████████████| 3.47MB / 3.47MB, 5.78MB/s  
New Data Upload               : 100%|██████████████| 3.47MB / 3.47MB, 5.78MB/s  
  yolov8n_int8.tflite         : 100%|██████████████| 3.47MB / 3.47MB            
Published: https://huggingface.co/unixio/nova-obstacle-detection
TFLite SHA-256: d2ccf2227b3fcc1b8cc7574